# 05｜Fleet：一次控制多台设备


> 运行方式：从上到下执行代码单元。控制类 Notebook **没有布尔安全开关**；执行控制单元就会把任务提交给 SDK。
> 需要仿真的课程请先在 FleetDock IDEA 中启动仿真，并确认 `drone1`、`drone2` 等设备在线。


本课使用 `drone1` 和 `drone2`。

![Fleet 多设备协同](assets/fleet.svg)


## 创建 Fleet

`FleetDock.core.fleet()` 把多个标准设备组成 Fleet。批量连接和 Fleet Plan 都会保留每台设备的结果。


In [ ]:
from fleetdock_runtime import FleetDock
from fleetdock import CapabilityName

FleetDock.init()
fleet = FleetDock.core.fleet(
    FleetDock.devices.drone1,
    FleetDock.devices.drone2,
    name="教学双机编队",
)

print(fleet.connect())
print("无人机成员：", [item.metadata.device_id for item in fleet.drones().devices])
print(
    "支持起飞的成员：",
    [item.metadata.device_id for item in fleet.by_capability(CapabilityName.TAKEOFF).devices],
)


In [ ]:
result = (
    FleetDock.core.plan(fleet)
    .for_drones()
    .wait_until_ready(timeout_s=90.0)
    .unlock()
    .takeoff(altitude=12.0)
    .return_to_launch()
    .run(step_timeout_s=120.0)
)

print(result)
print("成功设备：", result.succeeded_devices)
print("失败设备：", result.failed_devices)


In [ ]:
FleetDock.close()


## 重点

Fleet 不是简单循环。SDK 统一汇总成员、批量操作、事件和任务结果，业务代码不必自己管理每台设备的线程与异常。
